# W-algebras by quantum Hamiltonian reduction

`quantum_hamiltonian_reduction.py` constructs the universal W-algebra $W^k(\mathfrak g, f)$ of Kac–Roan–Wakimoto [KRW2003] and Kac–Wakimoto [KW2004]. The data are:

* a basic Lie superalgebra $\mathfrak g$: a `LieSuperalgebra` (see `02_lie_superalgebras.ipynb`). Its invariant form $(\cdot|\cdot)$, normalised as in the tables of [KW2004], is the form with respect to which the level is measured;
* an even nilpotent element $f \in \mathfrak g$, and a *good grading* for it: an element $x$ of the Cartan subalgebra whose eigenvalues $j$ under $\operatorname{ad} x$ are half-integers, $\mathfrak g = \bigoplus_j \mathfrak g_j$, with $f \in \mathfrak g_{-1}$ and $\operatorname{ad} f: \mathfrak g_j \to \mathfrak g_{j-1}$ injective for $j \ge 1/2$ and surjective for $j \le 1/2$ [KRW2003, EK2005]. The default is the Dynkin grading $x = h/2$ of an $\mathfrak{sl}_2$-triple $(e, h, f)$, which is always good;
* the level $k$: a number, or an element of a field such as $\mathbb Q(k)$.

`WAlgebra(R, g, f, k)` returns a `UniversalVertexAlgebra` over `R` (see `01_getting_started.ipynb`). It is freely generated by fields $J^{\{v\}}$, one for each $v$ in a basis of the centraliser $\mathfrak g^f = \{v : [f, v] = 0\}$, and $J^{\{v\}}$ has conformal weight $1 + j$ for $v \in \mathfrak g_{-j}$. The constructor computes the OPEs of these generators. With `conformal_vector='reduction'` it also chooses the conformal vector that the reduction provides; without it the algebra has the conformal weights (as `degrees`) but no conformal vector.

Since the result is a universal vertex algebra, everything the library does with universal vertex algebras applies to it: weight spaces, Jacobi identities, ideals and quotients, subalgebras, cosets, Zhu algebras, modules.

**Prerequisites:** `01_getting_started.ipynb`, `02_lie_superalgebras.ipynb`, `03_affine_vertex_algebras.ipynb`.

**Contents**

1. The Virasoro algebra from $\mathfrak{sl}_2$
2. $W_3$ from $\mathfrak{sl}_3$
3. Choosing $f$ and $x$
4. The nilpotent orbits of $\mathfrak{sl}_4$
5. Other Lie algebras
6. Superconformal algebras
7. How it is computed
8. Zhu algebras
9. Reduction of modules
10. The affine method, caching, pickling, limits

In [1]:
import vertex_algebras as va
from vertex_algebras import *
from vertex_algebras import LieSuperalgebra, WAlgebra, UniversalAffineVertexAlgebra
K.<k> = Frac(QQ['k'])

## 1. The Virasoro algebra from $\mathfrak{sl}_2$

The simplest reduction: $\mathfrak g = \mathfrak{sl}_2$ with its principal nilpotent, given as `'principal'` (up to conjugation it is the only nonzero nilpotent). The centraliser $\mathfrak g^f$ is spanned by $f$ itself, which lies in $\mathfrak g_{-1}$, so the W-algebra has a single generator of weight 2. It is the universal Virasoro algebra of central charge

$$c = 1 - \frac{6(k+1)^2}{k+2}.$$

The level is the variable `k` of the field $\mathbb Q(k)$ defined in the first cell. Without `names=`, the generators are called `J{v}` after the basis vectors $v$ of $\mathfrak g^f$, here `J{E[-alpha[1]]}` for $f = E_{-\alpha_1}$. The cell below prints the algebra, the weight of its generator, and checks the central charge.

In [2]:
sl2 = LieSuperalgebra(QQ, CartanMatrix(['A', 1]))
W = WAlgebra(K, sl2, 'principal', k, conformal_vector='reduction')
print(W)
print(W.degrees(), W.central_charge() == 1 - 6*(k + 1)^2/(k + 2))

Universal vertex algebra on generators ['J{E[-alpha[1]]}'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field
{'J{E[-alpha[1]]}': 2} True


`names=` gives the generators shorter names, in the order of increasing weight; here the one generator is called `L`.

The name should not suggest too much: the generator is the field $J^{\{f\}}$ as the reduction produces it, not the conformal vector. It is a multiple of the conformal vector $T$ (both have weight 2, and the weight-2 space is one-dimensional), and the cell below shows the factor, $T = -L/(k+2)$. Accordingly the OPE of $L$ with itself is a rescaled Virasoro OPE, while that of $T$ is the standard one, $[\partial T, 2T, 0, c/2]$.

In [3]:
Vir = WAlgebra(K, sl2, 'principal', k, names=['L'], conformal_vector='reduction')
L = Vir.gen('L')
T = Vir.conformal_vector()
print('L L ~', L.ope(L))
print('T =', T)
print('T T ~', T.ope(T))

L L ~ [(-k - 2)*dL, (-2*k - 4)*L, 0, -3*k^3 - 23/2*k^2 - 13*k - 4]
T = (-1/(k + 2))*L
T T ~ [(-1/(k + 2))*dL, (-2/(k + 2))*L, 0, (-3*k^2 - 11/2*k - 2)/(k + 2)]


The level can also be a number. A rational level gives an algebra over $\mathbb Q$, in which one can look for singular vectors and form quotients.

At $k = -2 + 2/5$ the central charge is $c = -22/5$. The function `singular_vectors` below finds the vectors of a given weight that are killed by $L_1 = T_{(2)}$ and $L_2 = T_{(3)}$; these two modes generate all positive modes, so such vectors are singular. There is one of weight 4. It generates the maximal ideal, and the quotient (`V.quotient`, see `06_subalgebras_ideals_quotients.ipynb`) is the simple Virasoro vertex algebra of the Lee–Yang minimal model [Wan1993]. The second cell compares the dimensions of its weight spaces with the Rogers–Ramanujan product $\prod_{n \equiv \pm 2 \bmod 5} (1 - q^n)^{-1}$.

In [4]:
LY = WAlgebra(QQ, sl2, 'principal', -2 + 2/5, names=['L'], conformal_vector='reduction')
print(LY.base_ring(), LY.central_charge())


def singular_vectors(V, wt):
    """The vectors of weight wt killed by T_(2) and T_(3) (the modes L_1, L_2)."""
    T = V.conformal_vector()
    B = V.weight_space(wt).basis()
    ims = [[T.nproduct(b, n) for n in (2, 3)] for b in B]
    keys = sorted(set((i, m) for im in ims for i, y in enumerate(im) for m in y.support()), key=str)
    M = matrix(V.base_ring(), [[im[i].monomial_coefficients().get(m, 0) for i, m in keys]
                               for im in ims])
    return [sum(c * b for c, b in zip(row, B)) for row in M.left_kernel().basis()]


v, = singular_vectors(LY, 4)
v

Rational Field -22/5


:L L: + 3/25*d^2L

In [5]:
Q = LY.quotient([v])
print([Q.weight_space(n).dimension() for n in range(13)])
R.<q> = PowerSeriesRing(ZZ)
print(prod(1 / (1 - q^n) for n in range(1, 13) if n % 5 in (2, 3)).add_bigoh(13))

[1, 0, 1, 1, 1, 1, 2, 2, 3, 3, 4, 4, 6]
1 + q^2 + q^3 + q^4 + q^5 + 2*q^6 + 2*q^7 + 3*q^8 + 3*q^9 + 4*q^10 + 4*q^11 + 6*q^12 + O(q^13)


## 2. $W_3$ from $\mathfrak{sl}_3$

The principal nilpotent of $\mathfrak{sl}_3$ has a two-dimensional centraliser, with basis vectors in $\mathfrak g_{-1}$ and $\mathfrak g_{-2}$. So the W-algebra has generators of weights 2 and 3, named `L` and `W` below: it is Zamolodchikov's $W_3$ algebra [Zam1985], with

$$c = 2 - \frac{24(k+2)^2}{k+3}.$$

The cell below prints the weights, checks the central charge, and prints the conformal vector $T$ (again a multiple of the generator `L`) and its OPE with the generator `W`.

In [6]:
sl3 = LieSuperalgebra(QQ, CartanMatrix(['A', 2]))
W3 = WAlgebra(K, sl3, 'principal', k, names=['L', 'W'], conformal_vector='reduction')
print(W3.degrees(), W3.central_charge() == 2 - 24*(k + 2)^2/(k + 3))
T, Wg = W3.conformal_vector(), W3.gen('W')
print('T =', T)
print('T W ~', T.ope(Wg))

{'L': 2, 'W': 3} True
T = (-1/(k + 3))*L
T W ~ [dW, 3*W, 2*k*L, 0, 24*k^3 + 94*k^2 + 90*k]


The OPE just printed has a third-order pole, $T_{(2)}W = 2kL \ne 0$: the generator of weight 3 that the reduction produces is some field of weight 3, not a primary one. The weight-3 space is spanned by $W$ and $\partial T$, so a primary field is obtained by adding a multiple of $\partial T$; the first lines of the cell below find the multiple and check that the corrected field $W'$ (`Wp`) is primary, $T_{(2)}W' = 0$ and $T_{(1)}W' = 3W'$.

The OPE of the primary field with itself is then Zamolodchikov's [Zam1985],

$$W'(z)W'(w) \sim N\Big(\frac{c/3}{(z-w)^6} + \frac{2T}{(z-w)^4} + \frac{\partial T}{(z-w)^3} + \frac{2\beta\Lambda + \frac{3}{10}\partial^2 T}{(z-w)^2} + \frac{\beta\partial\Lambda + \frac{1}{15}\partial^3 T}{z-w}\Big),$$

with $\Lambda = {:}TT{:} - \frac{3}{10}\partial^2 T$ and $\beta = 16/(22 + 5c)$ (compare `01_getting_started.ipynb`, section 2), up to a normalisation $N$ that depends on $k$. The cell reads off $N$ from the leading pole and compares all six poles.

Its last line checks the Jacobi identities of the generators. Every W-algebra satisfies them, so this is a test of the construction rather than of the algebra.

In [7]:
t = T.leading_support()
a = -T.nproduct(Wg, 2).monomial_coefficients()[t] / T.nproduct(T.derivative(), 2).monomial_coefficients()[t]
Wp = Wg + a * T.derivative()
print('T_(2) Wp =', T.nproduct(Wp, 2), '  T_(1) Wp = 3 Wp:', T.nproduct(Wp, 1) == 3 * Wp)
c = W3.central_charge()
N = 3 * Wp.nproduct(Wp, 5).monomial_coefficients()[()] / c
beta = 16 / (22 + 5 * c)
Lam = T.no(T) - 3/10 * T.derivative(2)
zam = [N * (beta * Lam.derivative() + T.derivative(3) / 15), N * (2 * beta * Lam + 3/10 * T.derivative(2)),
       N * T.derivative(), 2 * N * T, 0, N * c / 3 * W3.one()]
print('Zamolodchikov:', Wp.ope(Wp) == zam, '  N =', factor(N))
print('Jacobi identity:', W3.jacobi_identity_failures() == {})

T_(2) Wp = 0   T_(1) Wp = 3 Wp: True
Zamolodchikov: True   N = (-5/2) * (k + 4/3) * (k + 12/5) * (k + 3)^2
Jacobi identity: True


## 3. Choosing $f$ and $x$

The nilpotent `f` can be given in several ways:

* `'minimal'`: $f = E_{-\theta}$, for $\theta$ the root with $(\theta|\theta) = 2$ in the normalisation of $\mathfrak g$. These are the minimal W-algebras of [KW2004];
* `'principal'`: the principal nilpotent of the even part of $\mathfrak g$;
* a Jordan type on the natural representation, for $\mathfrak{sl}_n$, $\mathfrak{so}_n$, $\mathfrak{sp}_n$ (Cartan matrices in Bourbaki's labelling) and $\mathfrak{sl}(m|n)$, $\mathfrak{psl}(m|m)$, $\mathfrak{osp}(m|n)$, $\mathfrak{spo}(n|m)$: a partition, or a pair of partitions for the two parts of $\mathbb C^{m|n}$ (see `LieSuperalgebra.nilpotent_of_jordan_type`);
* a list of positive roots $\beta$, meaning $f = \sum E_{-\beta}$;
* an element of $\mathfrak g$.

The grading element $x$ must lie in the Cartan subalgebra, so that the root vectors are homogeneous. This is why the named nilpotents are built as sums of root vectors, chosen so that an $\mathfrak{sl}_2$-triple with $h$ in the Cartan subalgebra exists.

The cell below shows the pieces for $\mathfrak{sl}_3$. Its minimal nilpotent $E_{-\theta}$ and the nilpotent of Jordan type $(2, 1)$, which is $E_{-\alpha_1}$, are different elements of the same nilpotent orbit. For the second one the cell prints an $\mathfrak{sl}_2$-triple, the Dynkin grading $x = h/2$ with the eigenvalues of $\operatorname{ad} x$ on the basis of $\mathfrak g$, and a basis of the centraliser $\mathfrak g^f$.

In [8]:
f_min, f_21 = sl3.minimal_nilpotent(), sl3.nilpotent_of_jordan_type([2, 1])
print(f_min, '  ', f_21)
e, h, f = sl3.sl2_triple(f_21)
print('sl_2-triple: e =', e, '  h =', h)
x = sl3.dynkin_grading(f_21)
print('x =', x, ' ', sl3.ad_eigenvalues(x))
print('g^f:', sl3.centralizer_basis(f_21, x))

E[-alpha[1] - alpha[2]]    E[-alpha[1]]
sl_2-triple: e = E[alpha[1]]   h = h1
x = 1/2*h1   {'E[alpha[1]]': 1, 'E[alpha[2]]': -1/2, 'E[alpha[1] + alpha[2]]': 1/2, 'h1': 0, 'h2': 0, 'E[-alpha[1]]': -1, 'E[-alpha[2]]': 1/2, 'E[-alpha[1] - alpha[2]]': -1/2}
g^f: [h1 + 2*h2, E[alpha[2]], E[-alpha[1] - alpha[2]], E[-alpha[1]]]


Conjugate nilpotents give isomorphic W-algebras. Both of the above give the Bershadsky–Polyakov algebra [Pol1990, Ber1991]: a field of weight 1, two of weight $3/2$ and one of weight 2. As objects of the library they differ, since their generators are named after the bases of two different centralisers, but the weights, the central charges and the dimensions of the weight spaces agree, as the cell below shows.

In [9]:
BP, BP2 = WAlgebra(K, sl3, 'minimal', k, conformal_vector='reduction'), WAlgebra(K, sl3, [2, 1], k, conformal_vector='reduction')
print(BP.degrees())
print(BP2.degrees())
print(BP.central_charge() == BP2.central_charge(), factor(BP.central_charge()))
print([BP.weight_space(n/2).dimension() for n in range(9)])
print([BP2.weight_space(n/2).dimension() for n in range(9)])

{'J{h1 - h2}': 1, 'J{E[-alpha[1]]}': 3/2, 'J{E[-alpha[2]]}': 3/2, 'J{E[-alpha[1] - alpha[2]]}': 2}
{'J{h1 + 2*h2}': 1, 'J{E[alpha[2]]}': 3/2, 'J{E[-alpha[1] - alpha[2]]}': 3/2, 'J{E[-alpha[1]]}': 2}
True (-6) * (k + 3)^-1 * (k + 1/3) * (k + 3/2)
[1, 0, 1, 2, 3, 4, 8, 10, 17]
[1, 0, 1, 2, 3, 4, 8, 10, 17]


On the other hand, the same element $f$ described in different ways gives the same object (`is`). The principal nilpotent of $\mathfrak{sl}_3$ as `'principal'`, as the Jordan type $(3)$, as the list of the simple roots, and as an element:

In [10]:
(WAlgebra(K, sl3, 'principal', k, conformal_vector='reduction') is WAlgebra(K, sl3, [3], k, conformal_vector='reduction') is WAlgebra(K, sl3, [(1, 0), (0, 1)], k, conformal_vector='reduction')
 is WAlgebra(K, sl3, sl3.principal_nilpotent(), k, conformal_vector='reduction'))

True

The Dynkin grading is the default, but any good grading $x$ in the Cartan subalgebra can be given with `x=`; the good gradings of the simple Lie algebras are classified in [EK2005]. The conformal vector of the reduction depends on $x$, and with it the conformal weights and the central charge.

For the minimal nilpotent of $\mathfrak{sl}_3$ there is an even good grading, with $\alpha_1(x) = 1$ and $\alpha_2(x) = 0$. The cell below builds it, checks that it is good (`is_good_grading`), and constructs the Bershadsky–Polyakov algebra with it: all four generators now have integer weights, $1, 1, 2, 2$, and the central charge differs from the one above.

In [11]:
xe = (2 * sl3.h(1) + sl3.h(2)) / 3
print(sl3.ad_eigenvalues(xe), sl3.is_good_grading(xe, f_min))
BPe = WAlgebra(K, sl3, 'minimal', k, x=xe, conformal_vector='reduction')
print(BPe.degrees(), factor(BPe.central_charge()))

{'E[alpha[1]]': 1, 'E[alpha[2]]': 0, 'E[alpha[1] + alpha[2]]': 1, 'h1': 0, 'h2': 0, 'E[-alpha[1]]': -1, 'E[-alpha[2]]': 0, 'E[-alpha[1] - alpha[2]]': -1} True


{'J{h1 - h2}': 1, 'J{E[-alpha[2]]}': 1, 'J{E[-alpha[1]]}': 2, 'J{E[-alpha[1] - alpha[2]]}': 2} (-8) * (k + 3)^-1 * (k + 1) * (k + 3/2)


## 4. The nilpotent orbits of $\mathfrak{sl}_4$

The nilpotent orbits of $\mathfrak{sl}_n$ are labelled by the partitions of $n$, the Jordan types. For $\mathfrak{sl}_4$ the nonzero ones give four W-algebras:

| partition | nilpotent | W-algebra |
|---|---|---|
| $(4)$ | principal | $W_4$ |
| $(3, 1)$ | subregular | Feigin–Semikhatov's $W^{(2)}_4$ [FS2004] |
| $(2, 2)$ | rectangular | |
| $(2, 1, 1)$ | minimal | |

Kac–Roan–Wakimoto give a formula for the central charge of the reduction [KRW2003],

$$c(\mathfrak g, f, k) = \frac{k\,\mathrm{sdim}\,\mathfrak g}{k + h^\vee} - 12k(x|x) - \sum_{\alpha > 0} (-1)^{p(\alpha)} (12 m_\alpha^2 - 12 m_\alpha + 2) - \tfrac12 \mathrm{sdim}\,\mathfrak g_{1/2},$$

where the sum is over the roots $\alpha$ with $m_\alpha = \alpha(x) > 0$ and $p(\alpha)$ is the parity of $\alpha$. The function `krw_central_charge` below evaluates the formula from the Lie superalgebra alone. For each partition the cell prints the weights of the generators, whether the central charge of the computed algebra agrees with the formula, and the central charge in factored form.

In [12]:
def krw_central_charge(g, f, x):
    gr, B = g.ad_eigenvalues(x), g.basis()
    s = sum((-1)^B[a].is_even_odd() * (12*m^2 - 12*m + 2) for a, m in gr.items() if m > 0)
    sh = sum((-1)^B[a].is_even_odd() for a, m in gr.items() if m == 1/2)
    return (k * g.superdimension() / (k + g.dual_coxeter_number()) - 12 * k * g.invariant_form(x, x)
            - s - sh / 2)


sl4 = LieSuperalgebra(QQ, CartanMatrix(['A', 3]))
for lam in Partitions(4):
    if max(lam) == 1:
        continue
    W = WAlgebra(K, sl4, list(lam), k, conformal_vector='reduction')
    f = sl4.nilpotent_of_jordan_type(lam)
    x = sl4.dynkin_grading(f)
    print('%-12s weights %-32s KRW: %s   c = %s' % (lam, sorted(W.degrees().values()),
          W.central_charge() == krw_central_charge(sl4, f, x), factor(W.central_charge())))

[4]          weights [2, 3, 4]                        KRW: True   c = (-60) * (k + 4)^-1 * (k + 11/4) * (k + 16/5)


[3, 1]       weights [1, 2, 2, 2, 3]                  KRW: True   c = (-24) * (k + 4)^-1 * (k + 17/8) * (k + 8/3)
[2, 2]       weights [1, 1, 1, 2, 2, 2, 2]            KRW: True   c = (-12) * (k + 4)^-1 * (k^2 + 41/12*k + 8/3)


[2, 1, 1]    weights [1, 1, 1, 1, 3/2, 3/2, 3/2, 3/2, 2] KRW: True   c = (-6) * (k + 4)^-1 * k * (k + 3/2)


## 5. Other Lie algebras

Nothing above is special to type $A$. As a consistency check, take the isomorphism $\mathfrak{so}_5 \cong \mathfrak{sp}_4$. It maps long roots to long roots, so the normalised invariant forms agree and the level means the same on both sides. The W-algebras of corresponding nilpotent orbits must then have the same weights and central charges. The orbits correspond as $(5) \leftrightarrow (4)$ (principal; the algebra $W(2, 4)$, with generators of weights 2 and 4), $(3,1,1) \leftrightarrow (2,2)$ and $(2,2,1) \leftrightarrow (2,1,1)$ (minimal).

In [13]:
so5, sp4 = LieSuperalgebra(QQ, CartanMatrix(['B', 2])), LieSuperalgebra(QQ, CartanMatrix(['C', 2]))
for l5, l4 in [([5], [4]), ([3, 1, 1], [2, 2]), ([2, 2, 1], [2, 1, 1])]:
    W5, W4 = WAlgebra(K, so5, l5, k, conformal_vector='reduction'), WAlgebra(K, sp4, l4, k, conformal_vector='reduction')
    print('%-10s %-10s weights %-22s same c: %s   c = %s' % (l5, l4, sorted(W5.degrees().values()),
          W5.central_charge() == W4.central_charge(), factor(W5.central_charge())))

[5]        [4]        weights [2, 4]                 same c: True   c = (-60) * (k + 3)^-1 * (k + 13/6) * (k + 12/5)
[3, 1, 1]  [2, 2]     weights [1, 2, 2, 2]           same c: True   c = (-12) * (k + 3)^-1 * (k^2 + 8/3*k + 3/2)


[2, 2, 1]  [2, 1, 1]  weights [1, 1, 1, 3/2, 3/2, 2] same c: True   c = (-6) * (k + 3)^-1 * (k + 1/2) * (k + 1)


For the exceptional Lie algebras there is no natural representation with Jordan types, and Bala–Carter labels are not implemented. The other ways to give $f$ work: `'minimal'`, `'principal'`, a list of roots or an element. The cell below computes the minimal W-algebra of $G_2$, and shows the error raised when a Jordan type is asked for.

In [14]:
G2 = LieSuperalgebra(QQ, CartanMatrix(['G', 2]))
WG = WAlgebra(K, G2, 'minimal', k, conformal_vector='reduction')
print(sorted(WG.degrees().values()), factor(WG.central_charge()))
try:
    G2.nilpotent_of_jordan_type([3])
except ValueError as err:
    print('ValueError:', err)

[1, 1, 1, 3/2, 3/2, 3/2, 3/2, 2] (-6) * (k + 4)^-1 * k * (k + 5/3)
ValueError: Jordan types are defined for sl(m|n), psl(m|m), osp(m|n), spo(n|m) and Cartan matrices of type A-D (in Bourbaki's labelling)


## 6. Superconformal algebras

For a Lie superalgebra, $\mathfrak g_{\pm 1/2}$ may contain odd elements, and the minimal reduction then has odd generators of weight $3/2$: supercurrents. The minimal reductions of the following Lie superalgebras are the well-known superconformal algebras [KRW2003, KW2004]:

| $\mathfrak g$ | minimal W-algebra |
|---|---|
| $\mathfrak{spo}(2\vert 1) = \mathfrak{osp}(1\vert 2)$ | the Neveu–Schwarz algebra ($N = 1$) |
| $\mathfrak{sl}(2\vert 1)$ | the $N = 2$ superconformal algebra |
| $\mathfrak{spo}(2\vert 3) = \mathfrak{osp}(3\vert 2)$ | the $N = 3$ superconformal algebra |
| $\mathfrak{psl}(2\vert 2)$ | the small $N = 4$ superconformal algebra |
| $D(2,1;a)$ | the big $N = 4$ superconformal algebra |

The cell below constructs the first four. The Lie superalgebras are given by name (see `02_lie_superalgebras.ipynb`). For each it prints the weights of the generators and compares the central charge with the formula of section 4.

In [15]:
for name, g in [('N=1', LieSuperalgebra(QQ, 'spo(2|1)')), ('N=2', LieSuperalgebra(QQ, 'sl(2|1)')),
                ('N=3', LieSuperalgebra(QQ, 'spo(2|3)')), ('small N=4', LieSuperalgebra(QQ, 'psl(2|2)'))]:
    W = WAlgebra(K, g, 'minimal', k, conformal_vector='reduction')
    f = g.minimal_nilpotent()
    print('%-10s %-9s weights %-32s KRW: %s   c = %s' % (name, str(g).split()[2],
          sorted(W.degrees().values()),
          W.central_charge() == krw_central_charge(g, f, g.dynkin_grading(f)), factor(W.central_charge())))

N=1        spo(2|1)  weights [3/2, 2]                         KRW: True   c = (-6) * (k + 3/2)^-1 * (k + 1/2) * (k + 5/4)


N=2        sl(2|1)   weights [1, 3/2, 3/2, 2]                 KRW: True   c = (-6) * (k + 1/2)


N=3        spo(2|3)  weights [1, 1, 1, 3/2, 3/2, 3/2, 2]      KRW: True   c = (-6) * (k + 7/12)


small N=4  psl(2|2)  weights [1, 1, 1, 3/2, 3/2, 3/2, 3/2, 2] KRW: True   c = (-6) * (k + 1)


The big $N = 4$ algebra comes from the family $D(2,1;a)$, which depends on a parameter. The Lie superalgebra is defined over $\mathbb Q(a)$ and the level is a second variable, so the W-algebra lives over $\mathbb Q(a, k)$. It has six fields of weight 1 (the currents of $\mathfrak g^\natural = \mathfrak{sl}_2 \oplus \mathfrak{sl}_2$, the centraliser of the $\mathfrak{sl}_2$-triple), four of weight $3/2$ and one of weight 2. The central charge, $-6k - 3$, does not depend on $a$.

In [16]:
Ka.<a> = Frac(QQ['a'])
Kak = Frac(QQ['a', 'k'])
D = LieSuperalgebra(Ka, 'D(2,1;a)', a=a)
WD = WAlgebra(Kak, D, 'minimal', Kak.gen(1), conformal_vector='reduction')
print(sorted(WD.degrees().values()), factor(WD.central_charge()))

[1, 1, 1, 1, 1, 1, 3/2, 3/2, 3/2, 3/2, 2] (-3) * (2*k + 1)


The next two cells check that these algebras have the familiar OPEs, in the same way as for $W_3$ in section 2: the generators come with the normalisation of the reduction, which is read off and divided out.

**$N = 1$.** With $G$ the field of weight $3/2$, the Neveu–Schwarz algebra has

$$G(z)G(w) \sim \mu\Big(\frac{2c/3}{(z-w)^3} + \frac{2T(w)}{z-w}\Big)$$

for a normalisation $\mu$ of $G$; the cell finds $\mu$ from the first-order pole and compares.

In [17]:
N1 = WAlgebra(K, LieSuperalgebra(QQ, 'spo(2|1)'), 'minimal', k, names=['G', 'L'], conformal_vector='reduction')
G, T = N1.gen('G'), N1.conformal_vector()
t = T.leading_support()
mu = G.nproduct(G, 0).monomial_coefficients()[t] / (2 * T.monomial_coefficients()[t])
print(G.ope(G) == [2 * mu * T, 0, 2 * mu * N1.central_charge() / 3 * N1.one()], '  mu =', mu)
print(factor(N1.central_charge()))

True   mu = -1/2*k - 3/4
(-6) * (k + 3/2)^-1 * (k + 1/2) * (k + 5/4)


**$N = 2$.** The field $J$ of weight 1 is normalised so that the two fields $G^\pm$ of weight $3/2$ have charges $\pm 1$, that is $J_{(0)}G^\pm = \pm G^\pm$. Then $J(z)J(w) \sim \frac{c/3}{(z-w)^2}$, as in the $N = 2$ superconformal algebra. The generators are picked out by their weights, since their default names are long.

In [18]:
N2 = WAlgebra(K, LieSuperalgebra(QQ, 'sl(2|1)'), 'minimal', k, conformal_vector='reduction')
wts = N2.degrees()
J = N2.gen([s for s, w in wts.items() if w == 1][0])
Gp, Gm = [N2.gen(s) for s, w in wts.items() if w == 3/2]
charge = J.nproduct(Gp, 0).monomial_coefficients()[Gp.leading_support()]
J = J / charge
print('J_(0) G+ = G+:', J.nproduct(Gp, 0) == Gp, '  J_(0) G- = -G-:', J.nproduct(Gm, 0) == -Gm)
print('J J ~', J.ope(J), '  c/3 =', N2.central_charge() / 3)

J_(0) G+ = G+: True   J_(0) G- = -G-: True
J J ~ [0, -2*k - 1]   c/3 = -2*k - 1


Nilpotents other than the minimal one are given by Jordan types, as in section 3. For a Lie superalgebra the natural representation is $\mathbb C^{m|n}$, and a Jordan type is a pair of partitions, one for each part. Three examples: $\mathfrak{spo}(2|3) = \mathfrak{osp}(3|2)$ with the principal nilpotent of $\mathfrak{so}_3$ (types $(1,1)$ on $\mathbb C^2$ and $(3)$ on $\mathbb C^3$), the same superalgebra with the principal nilpotent of its even part (types $(2)$ and $(3)$), and $\mathfrak{sl}(3|1)$ with types $(3)$ and $(1)$. Again the central charges agree with the formula of section 4.

In [19]:
osp32, sl31 = LieSuperalgebra(QQ, 'spo(2|3)'), LieSuperalgebra(QQ, 'sl(3|1)')
for g, jt in [(osp32, ([1, 1], [3])), (osp32, ([2], [3])), (sl31, ([3], [1]))]:
    W = WAlgebra(K, g, jt, k, conformal_vector='reduction')
    f = g.nilpotent_of_jordan_type(*jt)
    print('%-9s %-14s weights %-28s KRW: %s   c = %s' % (str(g).split()[2], jt,
          sorted(W.degrees().values()),
          W.central_charge() == krw_central_charge(g, f, g.dynkin_grading(f)), factor(W.central_charge())))

spo(2|3)  ([1, 1], [3])  weights [1, 1, 1, 2, 2, 2]           KRW: True   c = (24) * (k + 1/12)


spo(2|3)  ([2], [3])     weights [3/2, 2, 2, 5/2]             KRW: True   c = (18) * (k + 1/3)


sl(3|1)   ([3], [1])     weights [1, 2, 2, 2, 3]              KRW: True   c = (-24) * (k + 2)^-1 * (k + 4/3) * (k + 13/8)


## 7. How it is computed

This section looks inside the construction; it is not needed to use `WAlgebra`.

By definition [KRW2003], $W^k(\mathfrak g, f)$ is the cohomology of a differential $d_{(0)}$ on the complex

$$C = V^k(\mathfrak g) \otimes F^{\mathrm{ch}} \otimes F^{\mathrm{ne}},$$

the tensor product of the universal affine vertex algebra, the charged ghosts (a pair for each basis vector of $\mathfrak g_{>0}$) and the neutral free superfermions on $\mathfrak g_{1/2}$ (see `04_ghost_systems.ipynb`, section 8). By Theorem 4.1 of [KRW2003] (and revision in [KW2004]), the cohomology is concentrated in charge 0, and it consists of the $d_{(0)}$-closed elements of charge degree 0 of a subalgebra $C^-$ of $C$.

The library follows this description. $C$ is a single universal vertex algebra; $C^-_0$ is its subalgebra generated by the dressed currents $J^{(a)}$, $a \in \mathfrak g_{\le 0}$, and the neutral fermions; and $W$ is the kernel of $d_{(0)}$ on $C^-_0$, computed weight space by weight space (a `ZeroModeKernel`, a kind of commutant; for commutants see `07_commutants_and_cosets.ipynb`).

`V.quantum_hamiltonian_reduction_complex(f)`, for an affine vertex algebra `V`, returns these pieces: `vertex_algebra()` is $C$, `differential()` is $d$, `subalgebra()` is $C^-_0$ and `kernel()` is $W$. For $\mathfrak{sl}_2$ the cell below prints the generators of $C$ (three currents and one pair of ghosts), the element $d$ with $d_{(0)}d = 0$, and the dimensions of the weight spaces of the kernel, which are those of the Virasoro algebra.

In [20]:
Q = UniversalAffineVertexAlgebra(K, sl2, level=k, conformal_vector='sugawara').quantum_hamiltonian_reduction_complex('principal')
print(Q)
C, d, Cm, Ker = Q.vertex_algebra(), Q.differential(), Q.subalgebra(), Q.kernel()
print(C.variable_names())
print('d =', d, '   d_(0) d =', d.nproduct(d, 0))
print(Ker)
print([Ker.weight_space(n).dimension() for n in range(9)])

Complex of the quantum Hamiltonian reduction of Universal affine vertex algebra of Lie superalgebra with Cartan matrix [[2]] and odd simple roots [] over Rational Field at level k over Fraction Field of Univariate Polynomial Ring in k over Rational Field by E[-alpha[1]]
('J{E[alpha[1]]}', 'J{h1}', 'J{E[-alpha[1]]}', 'phi{E[alpha[1]]}', 'phi*{E[alpha[1]]}')
d = :J{E[alpha[1]]} phi*{E[alpha[1]]}: + phi*{E[alpha[1]]}    d_(0) d = 0
Kernel of the zero modes of [:J{E[alpha[1]]} phi*{E[alpha[1]]}: + phi*{E[alpha[1]]}] on Vertex subalgebra of Universal vertex algebra on generators ['J{E[alpha[1]]}', 'J{h1}', 'J{E[-alpha[1]]}', 'phi{E[alpha[1]]}', 'phi*{E[alpha[1]]}'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field strongly generated by ['S0', 'S1']


[1, 0, 1, 1, 2, 2, 4, 4, 7]


The kernel is a subalgebra. That it can be replaced by a universal vertex algebra is the statement that the W-algebra is freely generated [KRW2003, DSK2005]: the normally ordered monomials in the generators $J^{\{v\}}$ and their derivatives form a basis. So the universal vertex algebra on these generators has weight spaces of the same dimensions as the kernel. For $W_3$:

In [21]:
Ker3 = UniversalAffineVertexAlgebra(K, sl3, level=k, conformal_vector='sugawara').quantum_hamiltonian_reduction_complex('principal').kernel()
print([Ker3.weight_space(n).dimension() for n in range(7)])
print([W3.weight_space(n).dimension() for n in range(7)])

[1, 0, 1, 2, 3, 4, 8]
[1, 0, 1, 2, 3, 4, 8]


Two steps remain. First, strong generators of the kernel are found and matched with a basis of $\mathfrak g^f$ by their linear parts, which gives the fields $J^{\{v\}}$. Second, the OPEs of these generators are computed. By default this is done through the Miura map of [KW2004], an injective homomorphism of $W$ into the much smaller algebra $V^{\tau}(\mathfrak g_0) \otimes F(\mathfrak g_{1/2})$, where products are cheap (`algorithm='miura'`). With `algorithm='complex'` the OPEs are computed in $C$ itself. The result is the same algebra:

In [22]:
W3c = WAlgebra(K, sl3, 'principal', k, names=['L', 'W'], algorithm='complex', conformal_vector='reduction')
print(W3c is W3)

True


## 8. Zhu algebras

Zhu's algebra $A(V)$ [Zhu1996] is an associative algebra whose modules correspond to the top spaces of the positive energy modules of $V$ (see `09_zhu_algebras.ipynb`). For W-algebras it has a classical meaning. De Sole and Kac [DSK2006] identify the finite W-algebra $U(\mathfrak g, f)$ with the **H-twisted** Zhu algebra of $W^k(\mathfrak g, f)$: the version of Zhu's algebra for the modules twisted by $e^{2\pi i L_0}$, on which the fields of half-integer weight have zero modes too. It is computed with `zhu_algebra(twist='H')` (see `09_zhu_algebras.ipynb`, section 10). If all weights are integers the twist is trivial and it is the usual Zhu algebra.

For $W_3$ the weights are integers. The finite W-algebra of a principal nilpotent is the centre of $U(\mathfrak g)$ [Kos1978], so the Zhu algebra is a polynomial algebra in the two classes $[L]$ and $[W]$. The cell below confirms that it is commutative: `relations()`, the dict of nonzero commutators of the generators, is empty. `is_known_exact()` says that the algebra computed is known to be the Zhu algebra, not only an algebra mapping onto it.

In [23]:
A = W3.zhu_algebra()
xL, xW = A.gens()
print(A, A.is_known_exact())
print('commutative:', xL * xW == xW * xL, '  nonzero commutators:', A.relations())

Zhu algebra of Universal vertex algebra on generators ['L', 'W'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field True
commutative: True   nonzero commutators: {}


With the even good grading of section 3, the Bershadsky–Polyakov algebra `BPe` has integer weights, so its Zhu algebra is again computed exactly. It is not commutative: it is generated by the classes of the four generators, and the cell below prints their nonzero commutators.

In [24]:
Ae = BPe.zhu_algebra()
print(Ae, Ae.is_known_exact())
for (p, q), r in Ae.relations().items():
    print('[[%s], [%s]] = %s' % (p, q, r))

Zhu algebra of Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field True
[[J{E[-alpha[2]]}], [J{h1 - h2}]] = -3*[J{E[-alpha[2]]}]
[[J{E[-alpha[1]]}], [J{h1 - h2}]] = 3*[J{E[-alpha[1]]}]
[[J{E[-alpha[1]]}], [J{E[-alpha[2]]}]] = (-1/3*k-1)*[J{h1 - h2}] + 1/3*[J{h1 - h2}]^2 + [J{E[-alpha[1] - alpha[2]]}]
[[J{E[-alpha[1] - alpha[2]]}], [J{E[-alpha[2]]}]] = -k*[J{E[-alpha[2]]}]
[[J{E[-alpha[1] - alpha[2]]}], [J{E[-alpha[1]]}]] = k*[J{E[-alpha[1]]}]


With the Dynkin grading, the Bershadsky–Polyakov algebra `BP` has the two generators $G^\pm$ of weight $3/2$, and the two kinds of Zhu algebra differ. The untwisted one sees only the classes $[J]$ and $[L]$ of the fields of integer weight, and the library does not claim that the result is exact (it prints as an "algebra mapping onto the Zhu algebra"). The H-twisted one has all four generators and is exact: it is the finite W-algebra of the minimal nilpotent of $\mathfrak{sl}_3$ [DSK2006]. In it $[J]$ acts on $[G^\pm]$ by $\pm 3$, and the commutator of $[G^+]$ and $[G^-]$ is quadratic in $[J]$ plus a multiple of $[L]$.

In [25]:
print(BP.zhu_algebra())
AB = BP.zhu_algebra(twist='H')
print(AB, AB.is_known_exact())
for (p, q), r in AB.relations().items():
    print('[[%s], [%s]] = %s' % (p, q, r))

Algebra mapping onto the Zhu algebra of Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[1]]}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field
H-twisted Zhu algebra of Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[1]]}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field True
[[J{E[-alpha[1]]}], [J{h1 - h2}]] = 3*[J{E[-alpha[1]]}]
[[J{E[-alpha[2]]}], [J{h1 - h2}]] = -3*[J{E[-alpha[2]]}]
[[J{E[-alpha[2]]}], [J{E[-alpha[1]]}]] = (1/4*k^2+5/8*k+3/8) + (1/6*k+1/2)*[J{h1 - h2}] - 1/3*[J{h1 - h2}]^2 - [J{E[-alpha[1] - alpha[2]]}]
[[J{E[-alpha[1] - alpha[2]]}], [J{E[-alpha[1]]}]] = (-1/2*k-3/2)*[J{E[-alpha[1]]}]
[[J{E[-alpha[1] - alpha[2]]}], [J{E[-alpha[2]]}]] = (1/2*k+3/2)*[J{E[-alpha[2]]}]


The centre of every finite W-algebra of $\mathfrak g$ is isomorphic to the centre of $U(\mathfrak g)$ [Pre2007]. For $\mathfrak{sl}_3$ that is a polynomial algebra on two Casimirs, of degrees 2 and 3. The cell below checks this in low degree by linear algebra: among the elements of filtration degree $\le 3$ of the H-twisted Zhu algebra, those commuting with all four generators form a space of dimension 3, spanned by $1$ and the two Casimirs.

In [26]:
from itertools import combinations_with_replacement
mons = [m for d in range(4) for m in combinations_with_replacement(range(4), d)
        if AB.degree_on_basis(m) <= 3]
els = [AB.monomial(m) for m in mons]
rows = []
for x in AB.gens():
    ims = [(x * e - e * x).monomial_coefficients() for e in els]
    keys = sorted(set(kk for d in ims for kk in d))
    rows += [[d.get(kk, 0) for d in ims] for kk in keys]
print('central elements of degree <= 3:', matrix(K, rows).right_kernel_matrix().nrows())

central elements of degree <= 3: 3


## 9. Reduction of modules

The reduction is also a functor on modules: a module $M$ of $V^k(\mathfrak g)$ gives a module $H(M)$ of $W^k(\mathfrak g, f)$. `quantum_hamiltonian_reduction_modules.py` implements it for highest weight modules: a Verma module $M(\lambda)$, built as the module induced from a Verma module of the Zhu algebra, or its simple quotient $L(\lambda)$ (see `10_modules.ipynb`). The method is `M.quantum_hamiltonian_reduction(f, reduction=...)`.

Unlike the W-algebra itself, $H(M)$ is not computed as a cohomology: the weight spaces of the complex $M \otimes F^{\mathrm{ch}} \otimes F^{\mathrm{ne}}$ are infinite-dimensional. Instead the library applies the theorems that determine $H(M)$, after checking their hypotheses. There are two reductions:

* `reduction='+'` (the default), with the differential used for the W-algebras above.
  * $H(M(\lambda))$ is a Verma module of $W$, for any good pair $(x, f)$ [KW2004, Thm 6.3].
  * For minimal $f$, $H(L(\lambda))$ is $0$ if $k - (\lambda|\theta) \in \mathbb Z_{\ge 0}$, and the simple quotient otherwise [Ara2005].
  * For principal $f$, $H(L(\lambda))$ is given under Arakawa's genericity condition [Ara2007, Thm 9.1.4].
* `reduction='-'`, for the principal nilpotent of a Lie algebra: $H(L(\lambda))$ is nonzero if and only if $\lambda$ is anti-dominant [Ara2007].

The highest weight of the reduced module, that is the eigenvalues of the zero modes of the generators of $W$ on its highest weight vector, is computed through the Miura map of section 7.

The first example has symbolic level and weight: the Verma module $M(\lambda)$ of affine $\mathfrak{sl}_2$ at level $k$ (the variables `lam` and `kk` below) reduces to the Virasoro Verma module of conformal weight

$$h = \frac{\lambda(\lambda+2)}{4(k+2)} - \frac{\lambda}{2}.$$

The highest weight is printed as the eigenvalue of the generator $J^{\{f\}}$, which is $-(k+2)$ times $L_0$ (section 1); `top_weight()` is the conformal weight $h$.

In [27]:
Kl.<kk, lam> = Frac(QQ['kk', 'lam'])
V2 = UniversalAffineVertexAlgebra(Kl, ['A', 1], level=kk)
M = V2.induced_module(V2.zhu_algebra().verma_module({'alphacheck[1]': lam}))
H = M.quantum_hamiltonian_reduction('principal')
print(H)
print(H.top_weight() == lam*(lam + 2)/(4*(kk + 2)) - lam/2, H.vertex_algebra().central_charge())

Verma module of highest weight {'J{E[-alpha[1]]}': 1/2*kk*lam - 1/4*lam^2 + 1/2*lam} over Universal vertex algebra on generators ['J{E[-alpha[1]]}'] over Fraction Field of Multivariate Polynomial Ring in kk, lam over Rational Field
True (-6*kk^2 - 11*kk - 4)/(kk + 2)


If only the highest weight is wanted, `V.quantum_hamiltonian_reduction_highest_weight(f, weight)` computes it without building any module. It returns the eigenvalues of the zero modes of the generators, and the conformal weight.

For $W_3$ and $\lambda = (a, b)$ in Dynkin labels, the cell below prints the eigenvalues of the zero modes of the generators `L` and `W` on the reduction of $M(\lambda)$, as polynomials in $k$, $a$, $b$, and checks that the conformal weight is that of [KW2004],

$$h = \frac{(\lambda|\lambda+2\rho)}{2(k+3)} - \lambda(x).$$

In [28]:
K3.<k3, a, b> = Frac(QQ['k3', 'a', 'b'])
V3 = UniversalAffineVertexAlgebra(K3, ['A', 2], level=k3)
values, h = V3.quantum_hamiltonian_reduction_highest_weight('principal', [a, b], names=['L', 'W'])
print(values)
print(h == (2*(a^2 + a*b + b^2)/3 + 2*a + 2*b)/(2*(k3 + 3)) - (a + b))

{'L': k3*a - 1/3*a^2 + k3*b - 1/3*a*b - 1/3*b^2 + 2*a + 2*b, 'W': -4/3*k3^2*a + 2/3*k3*a^2 - 2/27*a^3 - 2/3*k3^2*b + 1/3*k3*a*b - 1/9*a^2*b + 1/9*a*b^2 + 2/27*b^3 - 10/3*k3*a + 2/3*a^2 - 2/3*k3*b - 2/3*b^2 - 4/3*a + 4/3*b}
True


**The Lee–Yang model from $\mathfrak{sl}_2$.** Section 1 obtained the Lee–Yang vertex algebra as a quotient of the reduction at $k = -2 + 2/5$. Its modules are obtained by reducing modules of affine $\mathfrak{sl}_2$ at this level. The level is admissible, and the admissible weights are $\lambda = 0, -2/5, -4/5, -6/5, -8/5$.

The cell below reduces the five simple modules $L(\lambda)$ with both reductions. In each column the two modules of the Lee–Yang model, of conformal weights $h = 0$ and $h = -1/5$, appear twice each, and one module reduces to zero. The last line prints the central charge and the character of one of the reductions with $h = -1/5$; it is $q^{-1/5}$ times the other Rogers–Ramanujan product, $\prod_{n \equiv \pm 1 \bmod 5} (1 - q^n)^{-1}$.

In [29]:
from vertex_algebras.vertex_algebra_submodule import ZeroModule
VL = UniversalAffineVertexAlgebra(QQ, ['A', 1], level=-2 + 2/5)
AL = VL.zhu_algebra()


def simple(V, weight):
    return V.induced_module(V.zhu_algebra().verma_module(weight)).simple_quotient()


for l in [0, -2/5, -4/5, -6/5, -8/5]:
    L = simple(VL, {'alphacheck[1]': l})
    out = []
    for red in ('+', '-'):
        H = L.quantum_hamiltonian_reduction('principal', reduction=red)
        out.append('0' if isinstance(H, ZeroModule) else 'L(%s)' % H.top_weight())
    print('lambda = %5s:  H_+ = %-8s H_- = %s' % (l, out[0], out[1]))
H = simple(VL, {'alphacheck[1]': -4/5}).quantum_hamiltonian_reduction('principal')
print(H.vertex_algebra().central_charge(), H.character(6))

lambda =     0:  H_+ = L(0)     H_- = 0
lambda =  -2/5:  H_+ = L(-1/5)  H_- = L(0)
lambda =  -4/5:  H_+ = L(-1/5)  H_- = L(-1/5)
lambda =  -6/5:  H_+ = L(0)     H_- = L(-1/5)
lambda =  -8/5:  H_+ = 0        H_- = L(0)
-22/5 q^(-1/5) + q^(4/5) + q^(9/5) + q^(14/5) + 2*q^(19/5) + 2*q^(24/5) + 3*q^(29/5) + O(q^6)


**The 3-state Potts model from $\mathfrak{sl}_3$.** At $k + 3 = 4/5$ the principal reduction of $\mathfrak{sl}_3$ is $W_3$ at $c = 4/5$, the central charge of the 3-state Potts model. The cell below applies the "−" reduction to the simple modules $L(\lambda^I - (k+3)\lambda^F)$, where $\lambda^I$ is an integrable weight of level 1 and $\lambda^F$ a dominant weight of level $\le 4$. Each reduces to zero or to one of the modules of the $W_3$ minimal model; the cell counts how often each conformal weight occurs, and finds exactly $0, 1/15, 2/5, 2/3$.

In [30]:
VP = UniversalAffineVertexAlgebra(QQ, ['A', 2], level=-3 + 4/5)
tops = {}
for l0 in [(0, 0), (1, 0), (0, 1)]:
    for m1 in range(5):
        for m2 in range(5 - m1):
            lam_ = {'alphacheck[1]': l0[0] - 4/5*m1, 'alphacheck[2]': l0[1] - 4/5*m2}
            H = simple(VP, lam_).quantum_hamiltonian_reduction('principal', reduction='-', names=['L', 'W'])
            if not isinstance(H, ZeroModule):
                tops[H.top_weight()] = tops.get(H.top_weight(), 0) + 1
print(H.vertex_algebra().central_charge(), sorted(tops.items()))

4/5 [(0, 3), (1/15, 6), (2/5, 3), (2/3, 6)]


**Minimal reductions.** For minimal $f$ the vanishing criterion of [Ara2005] is easy to see in examples. Take the Bershadsky–Polyakov algebra ($\mathfrak{sl}_3$, minimal nilpotent) at $k = -3/2$, where $(\lambda|\theta) = \lambda_1 + \lambda_2$ in Dynkin labels. For five weights $\lambda$ the cell below prints $\lambda$, the number $k - \lambda_1 - \lambda_2$, and the reduction of $L(\lambda)$: it is the zero module exactly when that number is a non-negative integer.

In [31]:
VB = UniversalAffineVertexAlgebra(QQ, ['A', 2], level=-3/2)
for l in [(-1/2, -1/2), (-1/2, -1), (0, -3/2), (0, 0), (1/2, -2)]:
    H = simple(VB, {'alphacheck[1]': l[0], 'alphacheck[2]': l[1]}).quantum_hamiltonian_reduction('minimal')
    print(l, -3/2 - l[0] - l[1], H if isinstance(H, ZeroModule) else 'top weight %s' % H.top_weight())

(-1/2, -1/2) -1/2 top weight 0
(-1/2, -1) 0 Zero module over Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[1]]}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Rational Field
(0, -3/2) 0 Zero module over Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[1]]}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Rational Field
(0, 0) -3/2 top weight 0
(1/2, -2) 0 Zero module over Universal vertex algebra on generators ['J{h1 - h2}', 'J{E[-alpha[1]]}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1] - alpha[2]]}'] over Rational Field


**Characters.** The Euler–Poincaré character of the complex [KRW2003, Thm 3.1] gives the character of the reduction of a Verma module without computing the reduction; `V.quantum_hamiltonian_reduction_character(f, weight, prec)` evaluates it. It must agree with the character of the Verma module of $W$ with the reduced highest weight. The cell below compares the two for the $N = 2$ superconformal algebra (from $\mathfrak{sl}(2|1)$, at level $1/3$). The exponents increase in steps of $1/2$, because the modes $G^\pm_{-1/2}$ raise the weight by $1/2$.

In [32]:
VN = UniversalAffineVertexAlgebra(QQ, LieSuperalgebra(QQ, 'sl(2|1)'), level=1/3)
print(VN.quantum_hamiltonian_reduction_character('minimal', [1/2, 1/5], 3))
WN = VN.quantum_hamiltonian_reduction('minimal', conformal_vector='reduction')
print(WN.verma_module(VN.quantum_hamiltonian_reduction_highest_weight('minimal', [1/2, 1/5])[0]).character(3))

q^(-71/200) + 2*q^(29/200) + 3*q^(129/200) + 6*q^(229/200) + 11*q^(329/200) + 18*q^(429/200) + 28*q^(529/200) + O(q^3)
q^(-71/200) + 2*q^(29/200) + 3*q^(129/200) + 6*q^(229/200) + 11*q^(329/200) + 18*q^(429/200) + 28*q^(529/200) + O(q^3)


**What is not implemented.** The following cases are refused:

* the cohomology itself (a finite model of the complex would be needed);
* the "−" reduction for non-principal $f$ (it gives Ramond-twisted modules);
* W-algebras whose Verma modules have infinite-dimensional top spaces, which happens when $\mathfrak g^\natural$ is not abelian, for example the minimal reduction of $\mathfrak{sl}_4$;
* Weyl modules;
* simple modules for $f$ neither minimal nor principal, with the "+" reduction.

For example, the minimal reduction of a module of affine $\mathfrak{sl}_4$:

In [33]:
V4 = UniversalAffineVertexAlgebra(QQ, ['A', 3], level=1/2)
M4 = V4.induced_module(V4.zhu_algebra().verma_module({'alphacheck[%d]' % i: 0 for i in (1, 2, 3)}))
try:
    M4.quantum_hamiltonian_reduction('minimal')
except NotImplementedError as e:
    print(e)

the Verma modules of Universal vertex algebra on generators ['J{E[alpha[2]]}', 'J{h1 - h3}', 'J{h2}', 'J{E[-alpha[2]]}', 'J{E[-alpha[1]]}', 'J{E[-alpha[3]]}', 'J{E[-alpha[1] - alpha[2]]}', 'J{E[-alpha[2] - alpha[3]]}', 'J{E[-alpha[1] - alpha[2] - alpha[3]]}'] over Rational Field have infinite top levels (a generator of integer weight with nonzero weight under the intersection of h and g^f): not implemented yet


## 10. The affine method, caching, pickling, limits

`WAlgebra(R, g, f, k)` is also available as a method of the affine vertex algebra, which already knows $\mathfrak g$ and the level: `V.quantum_hamiltonian_reduction(f, ...)`, for the universal affine vertex algebra `V` of a `LieSuperalgebra` or of a Cartan type (see `03_affine_vertex_algebras.ipynb`). The results are cached and have unique representation: the cell below checks that the method returns the *same objects* as `W3` of section 2 and `N2` of section 6, and that a W-algebra saved and loaded again (`dumps`, `loads`) is the same object too.

In [34]:
V = UniversalAffineVertexAlgebra(K, ['A', 2], level=k, conformal_vector='sugawara')
print(V.quantum_hamiltonian_reduction('principal', names=['L', 'W'], conformal_vector='reduction') is W3)
V2 = UniversalAffineVertexAlgebra(K, LieSuperalgebra(QQ, 'sl(2|1)'), level=k, conformal_vector='sugawara')
print(V2.quantum_hamiltonian_reduction('minimal', conformal_vector='reduction') is N2)
print(loads(dumps(W3)) is W3)

True


True
True


**Limits.**

* The grading $x$ must lie in the Cartan subalgebra. A nilpotent element all of whose $\mathfrak{sl}_2$-triples have $h$ outside the Cartan subalgebra is refused with a `NotImplementedError`, as in the cell below for $f = E_{-\alpha_1} + E_{-\alpha_1 - \alpha_2}$ in $\mathfrak{sl}_3$. Give a conjugate nilpotent that is a suitable sum of root vectors instead (for example by its Jordan type), or give `x=` explicitly.
* Jordan types are available for the classical types only (section 5).
* The computation of the kernel (section 7) dominates the time: $W_4$ takes about a second, $W(G_2)$ about 20 s, $W_5$ about 100 s and $W(B_3)$ about 250 s.

In [35]:
f = sl3.root_vector((1, 0), -1) + sl3.root_vector((1, 1), -1)
try:
    WAlgebra(K, sl3, f, k, conformal_vector='reduction')
except NotImplementedError as err:
    print('NotImplementedError:', err)

NotImplementedError: the grading x must lie in the Cartan subalgebra: give f as a sum of root vectors, or x


## References

- [KRW2003] V. G. Kac, S.-S. Roan, M. Wakimoto, *Quantum reduction for affine superalgebras*, Comm. Math. Phys. 241 (2003), 307–342, [arXiv:math-ph/0302015](https://arxiv.org/abs/math-ph/0302015). — the quantum Hamiltonian reduction, the central charge (section 4), the structure theorem (section 7), the Euler–Poincaré character (section 9).
- [KW2004] V. G. Kac, M. Wakimoto, *Quantum reduction and representation theory of superconformal algebras*, Adv. Math. 185 (2004), 400–458; corrigendum, Adv. Math. 193 (2005), 453–455, [arXiv:math-ph/0304011](https://arxiv.org/abs/math-ph/0304011). — minimal W-algebras and superconformal algebras (sections 3, 6), the Miura map (section 7), the reduction of Verma modules (Thm 6.3, section 9).
- [EK2005] A. G. Elashvili, V. G. Kac, *Classification of good gradings of simple Lie algebras*, in: Lie groups and invariant theory, Amer. Math. Soc. Transl. Ser. 2, 213 (2005), 85–104, [arXiv:math-ph/0312030](https://arxiv.org/abs/math-ph/0312030). — good gradings (section 3).
- [Zam1985] A. B. Zamolodchikov, *Infinite additional symmetries in two-dimensional conformal quantum field theory*, Theoret. and Math. Phys. 65 (1985), 1205–1213, [doi:10.1007/BF01036128](https://doi.org/10.1007/BF01036128). — the $W_3$ algebra (section 2).
- [Wan1993] W. Wang, *Rationality of Virasoro vertex operator algebras*, Int. Math. Res. Not. 1993, no. 7, 197–211, [doi:10.1155/S1073792893000212](https://doi.org/10.1155/S1073792893000212). — the Virasoro minimal models as vertex algebras (section 1).
- [Pol1990] A. M. Polyakov, *Gauge transformations and diffeomorphisms*, Internat. J. Modern Phys. A 5 (1990), 833–842, [doi:10.1142/S0217751X90000386](https://doi.org/10.1142/S0217751X90000386). — the Bershadsky–Polyakov algebra (section 3).
- [Ber1991] M. Bershadsky, *Conformal field theories via Hamiltonian reduction*, Comm. Math. Phys. 139 (1991), 71–82, [doi:10.1007/BF02102729](https://doi.org/10.1007/BF02102729). — the Bershadsky–Polyakov algebra (section 3).
- [FS2004] B. L. Feigin, A. M. Semikhatov, *W⁽²⁾ₙ algebras*, Nuclear Phys. B 698 (2004), 409–449, [doi:10.1016/j.nuclphysb.2004.06.056](https://doi.org/10.1016/j.nuclphysb.2004.06.056). — the subregular W-algebra $W^{(2)}_4$ (section 4).
- [DSK2005] A. De Sole, V. G. Kac, *Freely generated vertex algebras and non-linear Lie conformal algebras*, Comm. Math. Phys. 254 (2005), 659–694, [arXiv:math-ph/0312042](https://arxiv.org/abs/math-ph/0312042). — free generation (section 7).
- [DSK2006] A. De Sole, V. G. Kac, *Finite vs affine W-algebras*, Jpn. J. Math. 1 (2006), 137–261, [arXiv:math-ph/0511055](https://arxiv.org/abs/math-ph/0511055). — finite W-algebras as H-twisted Zhu algebras (section 8).
- [Zhu1996] Y. Zhu, *Modular invariance of characters of vertex operator algebras*, J. Amer. Math. Soc. 9 (1996), 237–302, [doi:10.1090/S0894-0347-96-00182-8](https://doi.org/10.1090/S0894-0347-96-00182-8). — Zhu's algebra (section 8).
- [Kos1978] B. Kostant, *On Whittaker vectors and representation theory*, Invent. Math. 48 (1978), 101–184, [doi:10.1007/BF01390249](https://doi.org/10.1007/BF01390249). — the principal finite W-algebra is the centre of $U(\mathfrak g)$ (section 8).
- [Pre2007] A. Premet, *Enveloping algebras of Slodowy type and Joseph's ideal*, J. Eur. Math. Soc. 9 (2007), 487–543, [doi:10.4171/JEMS/86](https://doi.org/10.4171/JEMS/86). — the centre of a finite W-algebra (section 8).
- [Ara2005] T. Arakawa, *Representation theory of superconformal algebras and the Kac–Roan–Wakimoto conjecture*, Duke Math. J. 130 (2005), 435–478, [arXiv:math-ph/0405015](https://arxiv.org/abs/math-ph/0405015). — the reduction of simple modules for minimal $f$ (section 9).
- [Ara2007] T. Arakawa, *Representation theory of W-algebras*, Invent. Math. 169 (2007), 219–320, [arXiv:math/0506056](https://arxiv.org/abs/math/0506056). — the reduction of highest weight modules for principal $f$ (section 9).